In [39]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer 
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE


In [28]:
df = pd.read_csv("../data/raw/uc_diagnostic_tests.csv")
df

,mayo,plec,Wiek,witD,alat,albumina,APTT,ASPAT,Bialko calkowite,BILIRUBINA CAŁKOWITA,...,Morfologia RDW-CV,Morfologia RDW-SD,OB,PT (CZAS PROTROMBINOWY) INR,Potas,Sód,"TIBC,",UIBC,TRANSFERYNA,TRÓJGLICERYDY
0,2,0,36,"45,93",9.0,NaN,"22,29",17.0,"6,3","0,4",...,95,NaN,NaN,"0,888","4,88",139.0,NaN,NaN,NaN,NaN
1,0,1,69,"27,7",13.0,NaN,"28,2",15.0,"7,2","0,5",...,"73,2",NaN,NaN,"1,1","4,13",141.0,NaN,379.0,NaN,NaN
2,3,1,41,"48,8",28.0,NaN,"32,3",23.0,"6,8","0,4",...,"85,6",NaN,NaN,1,"4,34",141.0,353.0,242.0,NaN,118.0
3,1,1,31,"28,3",38.0,NaN,"34,9",42.0,"7,2","0,6",...,"66,3",NaN,NaN,"1,12","4,75",139.0,NaN,NaN,NaN,75.0
4,2,0,40,"27,8",14.0,"3,4","20,467",16.0,"6,2","0,3",...,"72,5",NaN,NaN,"0,965","4,66",139.0,NaN,294.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,1,0,75,NaN,15.0,"3,7","33,8",18.0,"6,7","0,4",...,"89,7",NaN,51.0,"1,05","4,57",133.0,302.0,201.0,NaN,73.0
248,3,1,69,NaN,22.0,NaN,"30,6",19.0,NaN,"1,1",...,"88,2",NaN,NaN,"0,93","4,17",145.0,280.0,198.0,NaN,144.0
249,3,1,23,NaN,15.0,NaN,35,19.0,"6,8","0,5",...,"79,7",NaN,NaN,"1,04","3,63",139.0,306.0,288.0,NaN,79.0
250,1,1,23,NaN,21.0,NaN,"31,3",25.0,"7,7","0,9",...,"86,6",NaN,NaN,"0,92","4,02",137.0,367.0,203.0,NaN,93.0


In [32]:
# 1. Data Preparation
# Perform data type conversion first
for col in df.select_dtypes(include=['object']).columns:
    if df[col].astype(str).str.contains(r'^\d+,\d+$', na=False).any():
        df[col] = df[col].str.replace(',', '.', regex=False).astype(float)

# Separate features (X) and target (y)
X = df.drop(columns="mayo")
y = df["mayo"]

# Now, define numeric_cols based on the feature matrix X (which no longer has 'mayo')
numeric_cols = X.select_dtypes(include=np.number).columns

# 2. Cross-validation Loop
results = []
folds = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42)

for fold, (train_idx, test_idx) in enumerate(folds.split(X, y)):
    # Split data for this fold
    X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Imputation (Fit on train, transform both)
    knn_imputer = KNNImputer(n_neighbors=5)
    X_train[numeric_cols] = knn_imputer.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols] = knn_imputer.transform(X_test[numeric_cols])

    # Scaling (Fit on train, transform both)
    scaler = StandardScaler()
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

    # SMOTE (on training data only)
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

    # Model Training
    rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    rf_classifier.fit(X_train_resampled, y_train_resampled)
    
    # Prediction and Evaluation
    y_pred = rf_classifier.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    results.append({
        "fold": fold,
        "precision": report["weighted avg"]["precision"],
        "recall": report["weighted avg"]["recall"],
        "f1-score": report["weighted avg"]["f1-score"],
        "balanced_accuracy": report["macro avg"]["recall"]
    })

# 3. Display Results
results_df = pd.DataFrame(results)
print("Leakage-free results:")
print(results_df)
print("\nMean Balanced Accuracy:", results_df['balanced_accuracy'].mean())

Leakage-free results:
   fold  precision    recall  f1-score  balanced_accuracy
0     0   0.443137  0.431373  0.421529           0.416278
1     1   0.317227  0.313725  0.292657           0.278361
2     2   0.468250  0.460000  0.458937           0.438131
3     3   0.316119  0.340000  0.313310           0.362962
4     4   0.411731  0.400000  0.400008           0.413159
5     5   0.392262  0.333333  0.343553           0.325952
6     6   0.360203  0.372549  0.359599           0.351593
7     7   0.457255  0.440000  0.445714           0.434975
8     8   0.220400  0.280000  0.244662           0.251040
9     9   0.316978  0.360000  0.328894           0.314388

Mean Balanced Accuracy: 0.35868384066913483


In [42]:
# --- Binary Classification Task ---

# 1. Data Preparation
# Create a copy to avoid modifying the original dataframe
df_binary = df.copy()

# Group Mayo scores into a binary target: (0, 1) -> 0 and (2, 3) -> 1
df_binary['severity_binary'] = df_binary['mayo'].apply(lambda x: 0 if x in [0, 1] else 1)

# Separate features (X) and the new binary target (y)
X_binary = df_binary.drop(columns=["mayo", "severity_binary"])
y_binary = df_binary['severity_binary']

# Identify numeric columns from the feature matrix
numeric_cols_binary = X_binary.select_dtypes(include=np.number).columns

# 2. Cross-validation Loop for Binary Task
results_binary = []
folds_binary = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42)

for fold, (train_idx, test_idx) in enumerate(folds_binary.split(X_binary, y_binary)):
    # Split data for this fold
    X_train, X_test = X_binary.iloc[train_idx].copy(), X_binary.iloc[test_idx].copy()
    y_train, y_test = y_binary.iloc[train_idx], y_binary.iloc[test_idx]

    # Imputation - increase max_iter to help convergence
    iterative_imputer = IterativeImputer(random_state=42, max_iter=30)
    X_train[numeric_cols_binary] = iterative_imputer.fit_transform(X_train[numeric_cols_binary])
    X_test[numeric_cols_binary] = iterative_imputer.transform(X_test[numeric_cols_binary])

    # Scaling
    scaler = StandardScaler()
    X_train[numeric_cols_binary] = scaler.fit_transform(X_train[numeric_cols_binary])
    X_test[numeric_cols_binary] = scaler.transform(X_test[numeric_cols_binary])

    # SMOTE
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

    # Model Training
    rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    rf_classifier.fit(X_train_resampled, y_train_resampled)
    
    # Prediction and Evaluation
    y_pred = rf_classifier.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    results_binary.append({
        "fold": fold,
        "precision": report["weighted avg"]["precision"],
        "recall": report["weighted avg"]["recall"],
        "f1-score": report["weighted avg"]["f1-score"],
        "balanced_accuracy": report["macro avg"]["recall"]
    })

# 3. Display Results for Binary Task
results_df_binary = pd.DataFrame(results_binary)
print("Leakage-free results for BINARY classification:")
print(results_df_binary)
print("\nMean Balanced Accuracy (Binary):", results_df_binary['balanced_accuracy'].mean())

/home/adam/programs/miniconda3/envs/uc_experiment/lib/python3.13/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/home/adam/programs/miniconda3/envs/uc_experiment/lib/python3.13/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/home/adam/programs/miniconda3/envs/uc_experiment/lib/python3.13/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/home/adam/programs/miniconda3/envs/uc_experiment/lib/python3.13/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/home/adam/programs/miniconda3/envs/uc_experiment/lib/python3.13/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping crite

Leakage-free results for BINARY classification:
   fold  precision    recall  f1-score  balanced_accuracy
0     0   0.729692  0.725490  0.716686           0.703762
1     1   0.579608  0.568627  0.570625           0.571317
2     2   0.654853  0.660000  0.650519           0.634647
3     3   0.763974  0.760000  0.760773           0.761364
4     4   0.690769  0.680000  0.681026           0.685065
5     5   0.629674  0.627451  0.628322           0.623041
6     6   0.635234  0.627451  0.629180           0.628527
7     7   0.589231  0.580000  0.582545           0.578818
8     8   0.720278  0.720000  0.715333           0.706169
9     9   0.587200  0.580000  0.581517           0.581169

Mean Balanced Accuracy (Binary): 0.647387669801463


In [ ]:
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline

# --- Hyperparameter Tuning with Nested Cross-Validation for Binary Task ---

# 1. Data Preparation (same as before)
df_binary = df.copy()
df_binary['severity_binary'] = df_binary['mayo'].apply(lambda x: 0 if x in [0, 1] else 1)
X_binary = df_binary.drop(columns=["mayo", "severity_binary"])
y_binary = df_binary['severity_binary']
numeric_cols_binary = X_binary.select_dtypes(include=np.number).columns

# 2. Create a Pipeline with SMOTE
# This pipeline chains all our preprocessing steps and the final classifier.
# We use the pipeline from imblearn to correctly handle SMOTE.
pipeline = ImbPipeline([
    ('imputer', IterativeImputer(random_state=42)),
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

# 3. Define the Hyperparameter Grid
# GridSearchCV will test all combinations of these parameters.
# The format is 'step_name__parameter_name'.
param_grid = {
    'imputer__n_nearest_features': [5, 10],  # Number of other features to use for imputation
    'imputer__max_iter': [15, 30],           # Iterations for the imputer
    'classifier__n_estimators': [100, 200], # Number of trees in the forest
    'classifier__max_depth': [10, 20, None], # Maximum depth of the trees
    'classifier__min_samples_split': [2, 5] # Minimum samples required to split a node
}

# 4. Nested Cross-Validation
results_tuned = []
outer_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42)

print("Starting nested cross-validation for hyperparameter tuning...")

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X_binary, y_binary)):
    print(f"--- Outer Fold {fold + 1}/10 ---")
    X_train, X_test = X_binary.iloc[train_idx].copy(), X_binary.iloc[test_idx].copy()
    y_train, y_test = y_binary.iloc[train_idx], y_binary.iloc[test_idx]

    # Inner CV for hyperparameter tuning (on the training set only)
    # We use a simple 3-fold CV for the inner loop to keep it fast.
    # 'scoring' is set to 'balanced_accuracy' to guide the search.
    # n_jobs=-1 uses all available CPU cores to speed up the search.
    grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='balanced_accuracy', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    # The best model from the grid search is automatically used for prediction
    y_pred = grid_search.predict(X_test)
    
    # Evaluate the best model on the outer loop's test set
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    results_tuned.append({
        "fold": fold,
        "balanced_accuracy": report["macro avg"]["recall"],
        "best_params": grid_search.best_params_
    })
    print(f"Best balanced accuracy in fold {fold + 1}: {report['macro avg']['recall']:.4f}")
    print(f"Best params found: {grid_search.best_params_}")


# 5. Display Final Tuned Results
results_df_tuned = pd.DataFrame(results_tuned)
print("\n--- Final Tuned Results ---")
print(results_df_tuned[['fold', 'balanced_accuracy']])
print(f"\nMean Balanced Accuracy after Tuning: {results_df_tuned['balanced_accuracy'].mean():.4f}")

# You can also inspect the best parameters found in each fold
print("\nBest parameters found across all folds:")
print(results_df_tuned[['fold', 'best_params']].to_string())
